# 01 - Data Cleaning

**Dataset:** UCI Statlog German Credit Data (1,000 rows, 20 attributes, Good/Bad credit-risk target)

ingestion + cleaning

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)

## 1. Load the raw data

The raw file (`german.data`) has no header row and no column names - just the
values, in the order documented in `german.names`. So supplying the column names
ourselves on load.

In [2]:
col_names = [
    "checking_status", "duration_months", "credit_history", "purpose",
    "credit_amount", "savings_status", "employment_since", "installment_rate",
    "personal_status_sex", "other_debtors", "residence_since", "property",
    "age", "other_installment_plans", "housing", "existing_credits",
    "job", "num_dependents", "telephone", "foreign_worker", "target"
]

df = pd.read_csv("../data/raw/german.data", header=None, names=col_names)
print(df.shape)
df.head()

(1000, 21)


,checking_status,duration_months,credit_history,purpose,credit_amount,savings_status,employment_since,installment_rate,personal_status_sex,other_debtors,residence_since,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker,target
0,A11,6,A34,A43,1169,A65,A75,4,A93,A101,4,A121,67,A143,A152,2,A173,1,A192,A201,1
1,A12,48,A32,A43,5951,A61,A73,2,A92,A101,2,A121,22,A143,A152,1,A173,1,A191,A201,2
2,A14,12,A34,A46,2096,A61,A74,2,A93,A101,3,A121,49,A143,A152,1,A172,2,A191,A201,1
3,A11,42,A32,A42,7882,A61,A74,2,A93,A103,4,A122,45,A143,A153,1,A173,2,A191,A201,1
4,A11,24,A33,A40,4870,A61,A73,3,A93,A101,4,A124,53,A143,A153,2,A173,2,A191,A201,2


## 2. Initial inspection

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   checking_status          1000 non-null   str  
 1   duration_months          1000 non-null   int64
 2   credit_history           1000 non-null   str  
 3   purpose                  1000 non-null   str  
 4   credit_amount            1000 non-null   int64
 5   savings_status           1000 non-null   str  
 6   employment_since         1000 non-null   str  
 7   installment_rate         1000 non-null   int64
 8   personal_status_sex      1000 non-null   str  
 9   other_debtors            1000 non-null   str  
 10  residence_since          1000 non-null   int64
 11  property                 1000 non-null   str  
 12  age                      1000 non-null   int64
 13  other_installment_plans  1000 non-null   str  
 14  housing                  1000 non-null   str  
 15  existing_credits

7 numeric columns and 13 categorical columns loaded, matching the dataset
documentation, plus the raw `target` column. Nothing parsed as an unexpected type,
so the load itself went fine.

## 3. Missing values and duplicate rows

In [4]:
print("Total missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Total missing values: 0
Duplicate rows: 0


No missing values, no duplicate rows. Worth checking explicitly rather than
assuming, even on a clean academic dataset like this one.

One thing to flag though: a few categorical columns have an "unknown / none"
option baked in (e.g. no checking account, no savings account).

## 4. Numeric vs categorical columns

In [5]:
numeric_cols = ["duration_months", "credit_amount", "installment_rate", "residence_since",
                "age", "existing_credits", "num_dependents"]
categorical_cols = [c for c in col_names if c not in numeric_cols + ["target"]]

print("Numeric:", numeric_cols)
print("\nCategorical:", categorical_cols)

Numeric: ['duration_months', 'credit_amount', 'installment_rate', 'residence_since', 'age', 'existing_credits', 'num_dependents']

Categorical: ['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment_since', 'personal_status_sex', 'other_debtors', 'property', 'other_installment_plans', 'housing', 'job', 'telephone', 'foreign_worker']


In [6]:
df[numeric_cols].describe()

,duration_months,credit_amount,installment_rate,residence_since,age,existing_credits,num_dependents
count,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000
mean,20.903000,3271.258000,2.973000,2.845000,35.546000,1.407000,1.155000
std,12.058814,2822.736876,1.118715,1.103718,11.375469,0.577654,0.362086
min,4.000000,250.000000,1.000000,1.000000,19.000000,1.000000,1.000000
25%,12.000000,1365.500000,2.000000,2.000000,27.000000,1.000000,1.000000
50%,18.000000,2319.500000,3.000000,3.000000,33.000000,1.000000,1.000000
75%,24.000000,3972.250000,4.000000,4.000000,42.000000,2.000000,1.000000
max,72.000000,18424.000000,4.000000,4.000000,75.000000,4.000000,2.000000


Ranges all look sane for a personal loan dataset - ages 19 to 75, durations 4 to
72 months, credit amounts 250 to 18,424 DM. No negative values or obvious typos, so
there's no numeric cleanup to do here beyond decoding the categories next.

## 5. Decode the categorical codes

The raw values are codes like `A11`, `A34`, `A93` - meaningless without the
dataset's attribute documentation. We map every code to a plain-English label so
everything downstream (charts, SQL, model output) is readable without a lookup
table.

In [7]:
checking_map = {"A11": "< 0 DM", "A12": "0 - 200 DM", "A13": ">= 200 DM", "A14": "no checking account"}
credit_history_map = {
    "A30": "no credits / all paid duly", "A31": "all credits at this bank paid duly",
    "A32": "existing credits paid duly", "A33": "delay in past", "A34": "critical / other credits existing"
}
purpose_map = {
    "A40": "car (new)", "A41": "car (used)", "A42": "furniture/equipment", "A43": "radio/tv",
    "A44": "domestic appliances", "A45": "repairs", "A46": "education", "A47": "vacation",
    "A48": "retraining", "A49": "business", "A410": "other"
}
savings_map = {
    "A61": "< 100 DM", "A62": "100 - 500 DM", "A63": "500 - 1000 DM",
    "A64": ">= 1000 DM", "A65": "unknown / no savings account"
}
employment_map = {
    "A71": "unemployed", "A72": "< 1 year", "A73": "1 - 4 years",
    "A74": "4 - 7 years", "A75": ">= 7 years"
}
personal_status_map = {
    "A91": "male: divorced/separated", "A92": "female: divorced/separated/married",
    "A93": "male: single", "A94": "male: married/widowed", "A95": "female: single"
}
other_debtors_map = {"A101": "none", "A102": "co-applicant", "A103": "guarantor"}
property_map = {
    "A121": "real estate", "A122": "building society/life insurance",
    "A123": "car or other property", "A124": "unknown / none"
}
other_installment_map = {"A141": "bank", "A142": "stores", "A143": "none"}
housing_map = {"A151": "rent", "A152": "own", "A153": "for free"}
job_map = {
    "A171": "unemployed/unskilled non-resident", "A172": "unskilled resident",
    "A173": "skilled employee/official", "A174": "management/self-employed/highly qualified"
}
telephone_map = {"A191": "none", "A192": "yes"}
foreign_worker_map = {"A201": "yes", "A202": "no"}

df["checking_status"] = df["checking_status"].map(checking_map)
df["credit_history"] = df["credit_history"].map(credit_history_map)
df["purpose"] = df["purpose"].map(purpose_map)
df["savings_status"] = df["savings_status"].map(savings_map)
df["employment_since"] = df["employment_since"].map(employment_map)
df["personal_status_sex"] = df["personal_status_sex"].map(personal_status_map)
df["other_debtors"] = df["other_debtors"].map(other_debtors_map)
df["property"] = df["property"].map(property_map)
df["other_installment_plans"] = df["other_installment_plans"].map(other_installment_map)
df["housing"] = df["housing"].map(housing_map)
df["job"] = df["job"].map(job_map)
df["telephone"] = df["telephone"].map(telephone_map)
df["foreign_worker"] = df["foreign_worker"].map(foreign_worker_map)

# if a code were missing from one of the dictionaries above, .map() would leave a NaN here - zero confirms every code in the raw file was accounted for
print("Missing values after decoding:", df.isnull().sum().sum())
df.head(3)

Missing values after decoding: 0


,checking_status,duration_months,credit_history,purpose,credit_amount,savings_status,employment_since,installment_rate,personal_status_sex,other_debtors,residence_since,property,age,other_installment_plans,housing,existing_credits,job,num_dependents,telephone,foreign_worker,target
0,< 0 DM,6,critical / other credits existing,radio/tv,1169,unknown / no savings account,>= 7 years,4,male: single,none,4,real estate,67,none,own,2,skilled employee/official,1,yes,yes,1
1,0 - 200 DM,48,existing credits paid duly,radio/tv,5951,< 100 DM,1 - 4 years,2,female: divorced/separated/married,none,2,real estate,22,none,own,1,skilled employee/official,1,none,yes,2
2,no checking account,12,critical / other credits existing,education,2096,< 100 DM,4 - 7 years,2,male: single,none,3,real estate,49,none,own,1,unskilled resident,2,none,yes,1


## 6. Recode the target

Raw target: `1` = Good, `2` = Bad. We turn this into a plain `risk` column and
drop the numeric version.

In [8]:
df["risk"] = df["target"].map({1: "Good", 2: "Bad"})
df = df.drop(columns=["target"])

df["risk"].value_counts()

risk
Good    700
Bad     300
Name: count, dtype: int64

## 7. Target distribution

700 Good, 300 Bad - a 70/30 split

In [9]:
df["risk"].value_counts(normalize=True).round(3) * 100

risk
Good    70.0
Bad     30.0
Name: proportion, dtype: float64

## 8. Feature engineering

Two small, easy-to-explain features:

- **`age_group`** - age bucketed into 5 ranges, for segment-level EDA (not fed into
  the model directly, since `age` already covers that signal).
- **`credit_per_month`** - `credit_amount / duration_months`. A rough stand-in for
  monthly repayment burden, since a 5,000 DM loan over 12 months is a very
  different commitment than the same amount over 48 months.

In [10]:
bins = [18, 25, 35, 45, 55, 76]
labels = ["19-25", "26-35", "36-45", "46-55", "56+"]
df["age_group"] = pd.cut(df["age"], bins=bins, labels=labels, right=True)

df["credit_per_month"] = (df["credit_amount"] / df["duration_months"]).round(2)

df[["age", "age_group", "credit_amount", "duration_months", "credit_per_month"]].head()

,age,age_group,credit_amount,duration_months,credit_per_month
0,67,56+,1169,6,194.83
1,22,19-25,5951,48,123.98
2,49,46-55,2096,12,174.67
3,45,36-45,7882,42,187.67
4,53,46-55,4870,24,202.92


## 9. Save the processed dataset

The file the EDA notebook

In [11]:
df.to_csv("../data/processed/german_credit_processed.csv", index=False)
print("Saved:", df.shape)

Saved: (1000, 23)


## Summary of cleaning decisions

- No missing values or duplicates, so no imputation or row dropping was needed.
- Categorical codes decoded into plain labels using the official attribute docs.
- "Unknown / none" categories (no checking account, no savings account, etc.) were
  kept as their own category instead of being dropped - not having one of these
  products is itself useful information for a credit decision.
- Target recoded from `1`/`2` to `Good`/`Bad`. Classes are imbalanced: 70% Good,
  30% Bad.
- Two engineered features added: `age_group` and `credit_per_month`.